# Patterns of Crime in Melbourne, Victoria

## KGG375 Geospatial Analysis

This project investigates the spatial distribution of recorded theft across Melbourne suburbs. It uses offence counts for the year ending March 2021 and population data from the 2021 Census.

Authors: Shaun Knight, Kalvin Pitt
Date: October 2026

## Study Area

The study area consists of suburbs within the Melbourne metropolitan region in Victoria, Australia. Suburb boundary data obtained from Vicmap is used as the spatial framework for analysis.

The aim of this project is to examine the geographical distribution of crime and identify whether spatial clustering exists between neighbouring suburbs.

The study area uses Melbourne's 31 metropolitan councils, excluding the part of Mitchell Shire included in some planning definitions. Suburbs are selected when their representative point falls inside these council boundaries. Full suburb polygons are retained.

Source: [Victorian Government, Metropolitan Melbourne](https://liveinmelbourne.vic.gov.au/discover/melbourne/metropolitan-melbourne).

## Data Acquisition and Preparation

Three datasets were used in this project:

1. Vicmap suburb boundary polygons.
2. Victorian crime statistics.
3. Suburb population data.

These datasets were imported into Python and prepared for subsequent spatial analysis.

## Suburb Boundary Dataset

The suburb boundary dataset was loaded as a GeoDataFrame using GeoPandas. Each polygon represents a Victorian suburb and contains identifying attributes including suburb names and administrative information.

In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Locate the existing boundary files
boundary_folder = Path("Data/VICMap Suburb Polys")
if not boundary_folder.exists():
    boundary_folder = Path("Data/VICMapSuburbPolys")
boundary_folder = boundary_folder / "mga2020_55/esrishape/user_polygon/user_polygon-0/VMADMIN"

suburbs = gpd.read_file(boundary_folder / "AD_LOCALITY_AREA_POLYGON.shp")
lga = gpd.read_file(boundary_folder / "AD_LGA_AREA_POLYGON.shp")

print(suburbs.head())
print(suburbs.columns)

In [ ]:
# Select Melbourne metropolitan councils
metro_names = [
    "BANYULE", "BAYSIDE", "BOROONDARA", "BRIMBANK", "CARDINIA",
    "CASEY", "DAREBIN", "FRANKSTON", "GLEN EIRA", "GREATER DANDENONG",
    "HOBSONS BAY", "HUME", "KINGSTON", "KNOX", "MANNINGHAM",
    "MARIBYRNONG", "MAROONDAH", "MELBOURNE", "MELTON", "MERRI-BEK",
    "MONASH", "MOONEE VALLEY", "MORNINGTON PENINSULA", "NILLUMBIK",
    "PORT PHILLIP", "STONNINGTON", "WHITEHORSE", "WHITTLESEA",
    "WYNDHAM", "YARRA", "YARRA RANGES"
]

metro = lga[lga["NAME"].isin(metro_names)].copy()

suburbs = suburbs[
    suburbs.representative_point().within(metro.geometry.union_all())
].copy()

## Population Dataset

Population data was obtained for Victorian suburbs and used to standardise crime counts into population-adjusted crime rates. Standardisation allows meaningful comparison between suburbs of different sizes.

In [ ]:
pop = pd.read_csv("Data/Suburb_pop.csv")
crime = pd.read_csv("Data/Crime_rates.csv")

print(pop.columns)
print(crime.columns)

## Crime Dataset

Crime data were obtained from the Victorian Crime Statistics Agency. The selected table contains offence counts recorded at the suburb or town level and includes offence classifications, local government areas, and crime counts.

Suburb-level data were selected because they provide a finer spatial resolution than local government area summaries and are therefore more suitable for spatial pattern analysis.

The selected offence subdivision is B40 Theft for the year ending March 2021 (1 April 2020 to 31 March 2021). The supplied CSV includes several reporting years, so the year and offence type are selected before aggregation.

In [ ]:
# Keep one reporting year and one offence type
crime = crime[
    (crime["Year"] == 2021)
    & (crime["Year ending"] == "March")
    & (crime["Offence Subdivision"] == "B40 Theft")
    & crime["Local Government Area"].str.upper().isin(metro_names)
].copy()

crime.head()

## Data Cleaning and Harmonisation

The crime, population, and boundary datasets were sourced from different organisations and therefore used different geographic definitions and naming conventions.

Prior to analysis, suburb names were standardised to ensure consistency between datasets. Records that could not be matched across all datasets were excluded from further analysis. Only suburbs present in the crime, population, and boundary datasets were retained for subsequent spatial analysis.

In [ ]:
# Remove the regional localities with the same names
pop = pop[
    ~pop["SAL_CODE"].isin(["SAL20199", "SAL21192"])
].copy()

suburbs["NAME"] = (
    suburbs["NUNIQ_LOC"]
    .str.upper()
    .str.strip()
)

pop["SUBURB NAME"] = (
    pop["SUBURB_NAME"]
    .str.upper()
    .str.replace(" (VIC.)", "", regex=False)
    .str.replace(" (BANYULE - VIC.)", "", regex=False)
    .str.replace(" (MELTON - VIC.)", "", regex=False)
    .str.strip()
)

crime["Suburb/Town Name"] = (
    crime["Suburb/Town Name"]
    .str.upper()
    .str.strip()
)

crime_suburbs = set(crime["Suburb/Town Name"].dropna())
pop_suburbs = set(pop["SUBURB NAME"].dropna())
poly_suburbs = set(suburbs["NAME"].dropna())

print("Crime suburbs:", len(crime_suburbs))
print("Population suburbs:", len(pop_suburbs))
print("Polygon suburbs:", len(poly_suburbs))

all_three = crime_suburbs & pop_suburbs & poly_suburbs

print("Matching suburbs:", len(all_three))

## Matching Geographic Units

After standardising suburb names, the overlap between datasets was assessed.

A total of 517 suburbs were present in all three datasets. Because spatial analysis requires both attribute and boundary information, only the common set of suburbs was retained.

Bellfield in Banyule and Hillside in Melton have different names in the Census and Vicmap files. Their names were matched, and the same-named regional Census localities were removed. Suburbs without matching records were excluded. A missing crime record was not assumed to mean zero offences.

In [ ]:
crime = crime[
    crime["Suburb/Town Name"].isin(all_three)
]

pop = pop[
    pop["SUBURB NAME"].isin(all_three)
]

suburbs = suburbs[
    suburbs["NAME"].isin(all_three)
]

print(len(crime))
print(len(pop))
print(len(suburbs))

## Aggregation of Crime Data

The crime dataset contained theft subcategories for each suburb. To enable suburb-level spatial analysis, offence counts were aggregated to produce a single total theft count for each suburb.

This aggregation ensured that each suburb was represented by one crime value that could be directly compared with population and spatial boundary datasets.

In [ ]:
# Remove commas before converting offence counts
crime["Offence Count"] = pd.to_numeric(
    crime["Offence Count"].astype(str).str.replace(",", "", regex=False),
    errors="coerce"
)

crime_summary = (
    crime
    .groupby("Suburb/Town Name", as_index=False)
    ["Offence Count"]
    .sum()
)

crime_summary.columns = [
    "SUBURB",
    "TOTAL_CRIME"
]

crime_summary.head()

In [ ]:
pop_summary = pop[
    ["SUBURB NAME", "SAL_CODE", "TOT_POP"]
].copy()

pop_summary.columns = [
    "SUBURB",
    "SAL_CODE",
    "POPULATION"
]

pop_summary["POPULATION"] = pd.to_numeric(
    pop_summary["POPULATION"],
    errors="coerce"
)

# Remove suburbs with very small populations
pop_summary = pop_summary[
    pop_summary["POPULATION"] > 500
]

## Merging Crime and Population Data

The aggregated crime dataset was joined with suburb population data using standardised suburb names. This enabled the calculation of population-adjusted crime rates, allowing meaningful comparisons between suburbs of different sizes.

In [ ]:
crime_pop = crime_summary.merge(
    pop_summary,
    on="SUBURB",
    how="inner"
)

crime_pop.head()

In [ ]:
print(len(crime_summary))
print(len(pop_summary))
print(len(crime_pop))

In [ ]:
crime_pop["CRIME_RATE_1000"] = (
    crime_pop["TOTAL_CRIME"]
    / crime_pop["POPULATION"]
) * 1000

crime_pop.head()

## Spatial Data Integration

The crime and population datasets were joined to the suburb boundary dataset using standardised suburb names. The resulting GeoDataFrame contains geometry, total crime counts, population, and crime rates for each suburb and forms the basis of the spatial analysis.

In [ ]:
crime_gdf = suburbs.merge(
    crime_pop,
    left_on="NAME",
    right_on="SUBURB",
    how="inner"
)

print(len(crime_gdf))
crime_gdf.head()

## Treatment of Small Population Areas

Initial analysis identified a number of suburbs with very small resident populations. Crime rates calculated for these locations were highly unstable because small population denominators can produce disproportionately large per-capita crime rates.

To reduce this effect, suburbs with populations of 500 residents or fewer were excluded. This reduced the matched dataset from 517 to 446 suburbs. The cutoff does not remove all population effects, especially in areas with many workers or visitors.

## Population and Crime Relationship

A Pearson correlation analysis was used to examine the relationship between suburb population and crime rate. This provides a measure of the strength and direction of association between population size and recorded crime rates.

This is an exploratory suburb-level comparison. The usual correlation p-value assumes independent observations, which nearby suburbs may not satisfy.

In [ ]:
from scipy.stats import pearsonr

corr, p = pearsonr(
    crime_gdf["POPULATION"],
    crime_gdf["CRIME_RATE_1000"]
)

print("Correlation coefficient:", corr)
print("P-value:", p)

In [ ]:
q95 = crime_gdf["CRIME_RATE_1000"].quantile(0.95)

hotspots = crime_gdf[
    crime_gdf["CRIME_RATE_1000"] >= q95
]

hotspots[
    ["SUBURB", "CRIME_RATE_1000"]
].sort_values(
    "CRIME_RATE_1000",
    ascending=False
).head(20)

## Hotspot Analysis

Suburbs with crime rates greater than or equal to the 95th percentile were flagged as hotspots in the descriptive table.

This approach highlights areas with exceptionally high rates relative to the study sample. The percentile flag does not test spatial clustering or establish statistically significant hotspots. Local Moran's I is used later for that comparison.

In [ ]:
print("95th percentile:", q95)

print("Number of hotspots:", len(hotspots))

print(
    "Percentage of suburbs:",
    round(
        len(hotspots) / len(crime_gdf) * 100,
        2
    ),
    "%"
)

In [ ]:
crime_gdf["HOTSPOT"] = (
    crime_gdf["CRIME_RATE_1000"] >= q95
)

crime_gdf.groupby("HOTSPOT")[
    ["POPULATION", "TOTAL_CRIME", "CRIME_RATE_1000"]
].describe()

In [ ]:
results_summary = pd.DataFrame({
    "Metric": [
        "Total suburbs",
        "Hotspot suburbs",
        "95th percentile threshold",
        "Population-crime correlation",
        "P-value"
    ],
    "Value": [
        len(crime_gdf),
        len(hotspots),
        round(q95, 2),
        round(corr, 3),
        round(p, 3)
    ]
})

results_summary

In [ ]:
hotspots[
    ["SUBURB", "POPULATION", "TOTAL_CRIME", "CRIME_RATE_1000"]
].sort_values(
    "CRIME_RATE_1000",
    ascending=False
)

## Interpretation of Hotspots

High recorded theft rates can reflect both the offence count and the size of the resident population. Commercial areas may also have large numbers of workers and visitors who are not included in the resident population.

The rates describe recorded offences in a suburb. They do not measure an individual resident's risk. The population correlation and percentile table provide initial comparisons, while maps show where the retained suburbs are located.

## Study Area Map

Blue polygons show suburbs retained for analysis. Grey areas are outside the sample. The boundaries use GDA2020 / MGA zone 55 (EPSG:7855), in metres.

In [ ]:
import matplotlib.pyplot as plt

# Create folders for the results
Path("figures").mkdir(exist_ok=True)
Path("outputs").mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 8))
metro.plot(ax=ax, color="#eeeeee", edgecolor="#999999", linewidth=0.4)
crime_gdf.plot(ax=ax, color="#547a96", edgecolor="white", linewidth=0.2)
ax.set_title("Melbourne study area and retained suburbs")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.ticklabel_format(style="plain", useOffset=False)
ax.annotate("N", xy=(0.93, 0.93), xytext=(0.93, 0.84),
            xycoords="axes fraction", ha="center",
            arrowprops=dict(arrowstyle="->", color="black"))
ax.text(0.02, 0.02, "Blue: included | Grey: outside analysis sample",
        transform=ax.transAxes, fontsize=8,
        bbox=dict(facecolor="white", alpha=0.9, edgecolor="none"))
fig.tight_layout()
fig.savefig("figures/study_area.png", dpi=160)
plt.show()

## Theft Rate Map

Rates have a strongly skewed distribution. A logarithmic colour scale is used so that
a few large rates do not hide the differences between most suburbs. The legend retains
the original units of recorded theft offences per 1,000 residents.

In [ ]:
from matplotlib.colors import LogNorm

fig, ax = plt.subplots(figsize=(8, 8))
metro.plot(ax=ax, color="#eeeeee", edgecolor="#bbbbbb", linewidth=0.3)
crime_gdf.plot(
    column="CRIME_RATE_1000", ax=ax, cmap="YlOrRd",
    norm=LogNorm(vmin=crime_gdf["CRIME_RATE_1000"].min(),
                 vmax=crime_gdf["CRIME_RATE_1000"].max()),
    legend=True, edgecolor="white", linewidth=0.2,
    legend_kwds={"label": "Theft offences per 1,000 residents", "shrink": 0.65}
)
ax.set_title("Recorded theft rates, year ending March 2021")
ax.set_axis_off()
fig.tight_layout()
fig.savefig("figures/theft_rates.png", dpi=160)
plt.show()

## Saving Results

Figures and tables are saved for the Jupyter Book. Running all cells recreates these outputs.

In [ ]:
# Save the tables used in the book
result_columns = ["SUBURB", "SAL_CODE", "POPULATION", "TOTAL_CRIME",
                  "CRIME_RATE_1000", "HOTSPOT"]

crime_gdf[result_columns].to_csv("outputs/theft_results.csv", index=False)
results_summary.to_csv("outputs/summary.csv", index=False)